# **Diplomado IA: Audio y Video - Parte 1**. <br> Práctico 5: Aplicaciones 2
---
---

**Profesores:**
- Alain Raymond
- Gabriel Sepúlveda
- Álvaro Soto

**Ayudante:**
- Andreina Cota
---
---

# **Instrucciones Generales**

El siguiente práctico se debe realizar de forma individual. El formato de entregar es el **archivo .ipynb con todas las celdas ejecutadas**. Las secciones donde se planteen preguntas de forma explícita, deben ser respondida en celdas de texto, y no se aceptará solo el _output_ de una celda de código como respuesta.

**Nombre alumno:** Roberto Araneda

El siguiente práctico cuenta con secciones que contienen los experimentos presentados durante la sesión de laboratorio, y actividades que deberán ser desarrolladas y luego entregadas como tarea. En esta oportunidad, las actividades corresponden a preguntas de alternativa.

Antes de responder, se recomienda **fuertemente** revisar las secciones previas donde se desarrollan los ejemplos, dado que algunas de las actividades pueden ser completadas reutilizando el mismo código.

**Fecha de entrega:** domingo 30 de agosto de 2026, 23:59 hrs.

#Sources

**End-to-End Audiovisual Speech Recognition**

dataset: https://www.robots.ox.ac.uk/~vgg/data/lip_reading/lrw1.html

paper: https://arxiv.org/pdf/1802.06424.pdf

github: https://github.com/mpc001/end-to-end-lipreading

#Preámbulo

In [1]:
import sys
import os
import os.path
import glob
import math
import random
import numpy as np
import cv2
import librosa
import errno
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F

In [2]:
!if [ ! -f Audiovisual.zip ]; then wget -q --show-progress https://www.dropbox.com/s/j6het8rq3j2ewng/Audiovisual.zip; fi
!if [ ! -f label_sorted.txt ]; then wget -q --show-progress https://www.dropbox.com/s/r44j8lhhsgjvjzb/label_sorted.txt; fi
!if [ ! -f lipread_testset_mini.tar.gz ]; then wget -q --show-progress https://www.dropbox.com/s/cbr5q72b8cef22i/lipread_testset_mini.tar.gz; fi
#!if [ ! -f lipread_testset.tar.gz ]; then wget -q --show-progress https://www.dropbox.com/s/4e3hkzaoizd491y/lipread_testset.tar.gz; fi
!unzip -q Audiovisual.zip
!tar xzf lipread_testset_mini.tar.gz
#!tar xzf lipread_testset.tar.gz

Audiovisual.zip     100%[===================>] 287.80M  61.4MB/s    in 5.0s    
label_sorted.txt    100%[===================>]   3.70K  --.-KB/s    in 0s      
lipread_testset_min 100%[===================>] 339.64M  78.1MB/s    in 4.1s    


#Preprocesamiento de datos

In [3]:
!cat label_sorted.txt | wc -l

500


In [4]:
!ls -l lipread_testset_mini/

total 2000
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ABOUT
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ABSOLUTELY
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ABUSE
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACCESS
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACCORDING
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACCUSED
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACROSS
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACTION
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ACTUALLY
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFFAIRS
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFFECTED
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFRICA
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFTER
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AFTERNOON
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGAIN
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGAINST
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGREE
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AGREEMENT
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 AHEAD
drwxrwxr-x 3 1000 1000 4096 Jan 20  2021 ALLEGATIONS
drwxrwxr-x

In [5]:
def extract_opencv(filename):
  video = []
  cap = cv2.VideoCapture(filename)
  while cap.isOpened():
    ret, frame = cap.read() # BGR
    if ret:
      video.append(frame)
    else:
      break
  cap.release()
  video = np.array(video)
  return video[...,::-1]

def video_converter(basedir, basedir_to_save):
  if not os.path.isdir( basedir_to_save ):
    os.makedirs( basedir_to_save, exist_ok = True )
  filenames = glob.glob(os.path.join(basedir, '*', '*', '*.mp4')) # <basedir>/<word>/<train, val, test>/<filename.mp4>
  for filename in filenames:
    data = extract_opencv(filename)[:, 115:211, 79:175]
    path_to_save = os.path.join(basedir_to_save,
                  filename.split('/')[-3],
                  filename.split('/')[-2],
                  filename.split('/')[-1][:-4]+'.npz')
    if not os.path.exists(os.path.dirname(path_to_save)):
      try:
        os.makedirs(os.path.dirname(path_to_save))
      except OSError as exc:
        if exc.errno != errno.EEXIST:
          raise
    np.savez(path_to_save, data=data)

def audio_converter(basedir, basedir_to_save):
  if not os.path.isdir( basedir_to_save ):
    os.makedirs( basedir_to_save, exist_ok = True )
  filenames = glob.glob(os.path.join(basedir, '*', '*', '*.mp4')) # <basedir>/<word>/<train, val, test>/<filename.mp4>
  for filename in filenames:
    data = librosa.load(filename, sr=16000)[0][-19456:]
    path_to_save = os.path.join(basedir_to_save,
                  filename.split('/')[-3],
                  filename.split('/')[-2],
                  filename.split('/')[-1][:-4]+'.npz')
    if not os.path.exists(os.path.dirname(path_to_save)):
      try:
        os.makedirs(os.path.dirname(path_to_save))
      except OSError as exc:
        if exc.errno != errno.EEXIST:
          raise
    np.savez( path_to_save, data=data)

In [8]:
import warnings
import logging

# El fallback PySoundFile → audioread emite 2 warnings por archivo.
# Los silencio solo dentro de este bloque, para no ocultar warnings del resto del notebook.
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    logging.getLogger('libav').setLevel(logging.ERROR)

    video_converter('lipread_testset_mini', 'preprocessed/video')
    audio_converter('lipread_testset_mini', 'preprocessed/audio')

# --- verificación ---
import numpy as np, glob

v = sorted(glob.glob('preprocessed/video/*/*/*.npz'))
a = sorted(glob.glob('preprocessed/audio/*/*/*.npz'))
print(f'✓ preprocesamiento completo: {len(v)} clips de video | {len(a)} clips de audio')

dv = np.load(v[0])['data']
da = np.load(a[0])['data']
print(f'  video: {dv.shape} {dv.dtype}   (esperado: (29, 96, 96, 3) uint8)')
print(f'  audio: {da.shape} {da.dtype}   (esperado: (19456,) float32)')

# clips que romperían el forward de la celda 27
malos_v = [f for f in v if np.load(f)['data'].shape[0] != 29]
malos_a = [f for f in a if np.load(f)['data'].shape[0] != 19456]
print(f'  clips de video con != 29 frames:      {len(malos_v)}')
print(f'  clips de audio con != 19456 muestras: {len(malos_a)}')
if malos_v or malos_a:
    print('  ⚠️', (malos_v + malos_a)[:5])

# distribución de clases del mini test set
from collections import Counter
print('\n  clips por palabra:', dict(Counter(f.split('/')[-3] for f in v)))


✓ preprocesamiento completo: 2500 clips de video | 2500 clips de audio
  video: (29, 96, 96, 3) uint8   (esperado: (29, 96, 96, 3) uint8)
  audio: (19456,) float32   (esperado: (19456,) float32)
  clips de video con != 29 frames:      0
  clips de audio con != 19456 muestras: 0

  clips por palabra: {'ABOUT': 5, 'ABSOLUTELY': 5, 'ABUSE': 5, 'ACCESS': 5, 'ACCORDING': 5, 'ACCUSED': 5, 'ACROSS': 5, 'ACTION': 5, 'ACTUALLY': 5, 'AFFAIRS': 5, 'AFFECTED': 5, 'AFRICA': 5, 'AFTER': 5, 'AFTERNOON': 5, 'AGAIN': 5, 'AGAINST': 5, 'AGREE': 5, 'AGREEMENT': 5, 'AHEAD': 5, 'ALLEGATIONS': 5, 'ALLOW': 5, 'ALLOWED': 5, 'ALMOST': 5, 'ALREADY': 5, 'ALWAYS': 5, 'AMERICA': 5, 'AMERICAN': 5, 'AMONG': 5, 'AMOUNT': 5, 'ANNOUNCED': 5, 'ANOTHER': 5, 'ANSWER': 5, 'ANYTHING': 5, 'AREAS': 5, 'AROUND': 5, 'ARRESTED': 5, 'ASKED': 5, 'ASKING': 5, 'ATTACK': 5, 'ATTACKS': 5, 'AUTHORITIES': 5, 'BANKS': 5, 'BECAUSE': 5, 'BECOME': 5, 'BEFORE': 5, 'BEHIND': 5, 'BEING': 5, 'BELIEVE': 5, 'BENEFIT': 5, 'BENEFITS': 5, 'BETTER': 5

In [7]:
import numpy as np, glob
v = sorted(glob.glob('preprocessed/video/*/*/*.npz'))
a = sorted(glob.glob('preprocessed/audio/*/*/*.npz'))
print(f'video: {len(v)} clips | audio: {len(a)} clips')

dv = np.load(v[0])['data']; da = np.load(a[0])['data']
print('video:', dv.shape, dv.dtype, '| audio:', da.shape, da.dtype)

# ¿algún clip quedó corto? esto es lo que reventaría en la celda 27
malos_v = [f for f in v if np.load(f)['data'].shape[0] != 29]
malos_a = [f for f in a if np.load(f)['data'].shape[0] != 19456]
print('clips de video != 29 frames:', len(malos_v))
print('clips de audio != 19456 muestras:', len(malos_a))


video: 2500 clips | audio: 2500 clips
video: (29, 96, 96, 3) uint8 | audio: (19456,) float32
clips de video != 29 frames: 0
clips de audio != 19456 muestras: 0


#Dataloader

In [9]:
def load_audio_file(filename):
  return np.load(filename)['data']

def load_video_file(filename):
  cap = np.load(filename)['data']
  arrays = np.stack([cv2.cvtColor(cap[_], cv2.COLOR_RGB2GRAY) for _ in range(29)], axis=0)
  arrays = arrays / 255.
  return arrays

class MyDataset():
  def __init__(self, folds, audio_path, video_path):
    '''
      folds: partition type -> test, train, val
      audio_path: ruta a archivos de audio numpy
      video_path: ruta a archivos de video numpy
    '''
    self.folds = folds
    self.audio_path = audio_path
    self.video_path = video_path
    self.clean = 1 / 7.
    with open('label_sorted.txt') as myfile:
      self.data_dir = myfile.read().splitlines()
    self.filenames = glob.glob(os.path.join(self.audio_path, '*', self.folds, '*.npz'))
    self.list = {}
    for i, x in enumerate(self.filenames):
      target = x.split('/')[-3]
      for j, elem in enumerate(self.data_dir):
        if elem == target:
          self.list[i] = [x]
          self.list[i].append(j)

  def normalisation(self, inputs):
    inputs_std = np.std(inputs)
    if inputs_std == 0.:
      inputs_std = 1.
    return (inputs - np.mean(inputs))/inputs_std

  def __getitem__(self, idx):
    video_inputs = load_video_file(os.path.join(self.video_path,
                          self.list[idx][0].split('/')[-3],
                          self.list[idx][0].split('/')[-2],
                          self.list[idx][0].split('/')[-1][:-4]+'.npz'))
    self.list[idx][0] = self.list[idx][0]
    audio_inputs = load_audio_file(self.list[idx][0])
    audio_inputs = self.normalisation(audio_inputs)
    labels = self.list[idx][1]
    return audio_inputs, video_inputs, labels

  def __len__(self):
    return len(self.filenames)

In [10]:
def data_loader(datatype, audio_dataset, video_dataset, batch_size):
  dsets = MyDataset(datatype, audio_dataset, video_dataset)
  dset_loader = torch.utils.data.DataLoader(dsets, batch_size = batch_size, shuffle=True, num_workers=4)
  dset_size = len(dsets)
  print('\nStatistics: {}: {}'.format(datatype, dset_size))
  return dset_loader, dset_size

#Modelo

##Bloques genéricos

In [11]:
class GRU(nn.Module):

  def __init__(self, input_size, hidden_size, num_layers, num_classes, output_layer=False, every_frame=False):
    super(GRU, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.output_layer = output_layer
    self.every_frame = every_frame
    self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
    self.fc = nn.Linear(hidden_size*2, num_classes)

  def to( self, device ):
    self.device = device
    return super( GRU, self ).to( device )

  def forward(self, x):
    h0 = Variable(torch.zeros(self.num_layers*2, x.size(0), self.hidden_size).to(self.device))
    # Forward propagate RNN
    out, _ = self.gru(x, h0)
    if self.output_layer:
      if self.every_frame:
        out = self.fc(out)  # predictions based on every time step
      else:
        out = self.fc(out[:, -1, :])  # predictions based on last time-step
    return out

##Modelo de audio


In [12]:
class BasicBlock1D(nn.Module):

  def __init__(self, inplanes, planes, stride=1, downsample=None):
    super(BasicBlock1D, self).__init__()
    self.downsample = downsample
    self.stride = stride

    self.conv1 = nn.Conv1d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm1d(planes)
    self.relu = nn.ReLU(inplace=True)

    self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm1d(planes)

  def forward(self, x):
    residual = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
      residual = self.downsample(x)
    out += residual
    out = self.relu(out)
    return out


class ResNet(nn.Module):

  def __init__(self, block, layers, num_classes=1000):
    self.inplanes = 64
    super(ResNet, self).__init__()
    self.layer1 = self._make_layer(block, 64, layers[0])
    self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
    self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
    self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
    self.avgpool = nn.AvgPool1d(kernel_size=21, padding=1)
    self.fc = nn.Linear(512, num_classes)

  def _make_layer(self, block, planes, blocks, stride=1):
    downsample = None
    if stride != 1 or self.inplanes != planes:
      downsample = nn.Sequential(
        nn.Conv1d(self.inplanes, planes,
              kernel_size=1, stride=stride, bias=False),
        nn.BatchNorm1d(planes),
      )

    layers = []
    layers.append(block(self.inplanes, planes, stride, downsample))
    self.inplanes = planes
    for i in range(1, blocks):
      layers.append(block(self.inplanes, planes))

    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = x.transpose(1, 2)
    x = x.contiguous()
    x = x.view(-1, x.size(2))
    x = self.fc(x)
    return x


class AudioLipreading(nn.Module):
  def __init__(self, inputDim=256, hiddenDim=512, nClasses=500, frameLen=29):
    super(AudioLipreading, self).__init__()
    self.inputDim = inputDim
    self.hiddenDim = hiddenDim
    self.nClasses = nClasses
    self.frameLen = frameLen
    self.nLayers = 2
    # frontend1D
    self.fronted1D = nn.Sequential(
        nn.Conv1d(1, 64, kernel_size=80, stride=4, padding=38, bias=False),
        nn.BatchNorm1d(64),
        nn.ReLU(True)
        )
    # resnet
    self.resnet18 = ResNet(BasicBlock1D, [2, 2, 2, 2], num_classes=self.inputDim)
    # backend_gru
    self.gru = GRU(self.inputDim, self.hiddenDim, self.nLayers, self.nClasses)

  def to( self, device ):
    self.gru.to(device)
    return super( AudioLipreading, self ).to( device )

  def forward(self, x):
    x = x.view(-1, 1, x.size(1))
    x = self.fronted1D(x)
    x = x.contiguous()
    x = self.resnet18(x)
    x = x.view(-1, self.frameLen, self.inputDim)
    x = self.gru(x)
    return x

##Modelo de video

In [13]:
class BasicBlock2D(nn.Module):

  def __init__(self, inplanes, planes, stride=1, downsample=None):
    super(BasicBlock2D, self).__init__()
    self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(planes)
    self.relu = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(planes)
    self.downsample = downsample
    self.stride = stride

  def forward(self, x):
    residual = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
      residual = self.downsample(x)
    out += residual
    out = self.relu(out)
    return out


class ResNet2D(nn.Module):

  def __init__(self, block, layers, num_classes=1000):
    self.inplanes = 64
    super(ResNet2D, self).__init__()
    self.layer1 = self._make_layer(block, 64, layers[0])
    self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
    self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
    self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
    self.avgpool = nn.AvgPool2d(2)
    self.fc = nn.Linear(512, num_classes)
    self.bnfc = nn.BatchNorm1d(num_classes)

  def _make_layer(self, block, planes, blocks, stride=1):
    downsample = None
    if stride != 1 or self.inplanes != planes:
      downsample = nn.Sequential(
        nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride, bias=False),
        nn.BatchNorm2d(planes),
      )

    layers = []
    layers.append(block(self.inplanes, planes, stride, downsample))
    self.inplanes = planes
    for i in range(1, blocks):
      layers.append(block(self.inplanes, planes))

    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = x.view(x.size(0), -1)
    x = self.fc(x)
    x = self.bnfc(x)
    return x


class VideoLipreading(nn.Module):

  def __init__(self, inputDim=256, hiddenDim=512, nClasses=500, frameLen=29):
    super(VideoLipreading, self).__init__()
    self.inputDim = inputDim
    self.hiddenDim = hiddenDim
    self.nClasses = nClasses
    self.frameLen = frameLen
    self.nLayers = 2
    # frontend3D
    self.frontend3D = nn.Sequential(
        nn.Conv3d(1, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
        nn.BatchNorm3d(64),
        nn.ReLU(True),
        nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        )
    # resnet
    self.resnet34 = ResNet2D(BasicBlock2D, [3, 4, 6, 3], num_classes=self.inputDim)
    # backend_gru
    self.gru = GRU(self.inputDim, self.hiddenDim, self.nLayers, self.nClasses)

  def to( self, device ):
    self.gru.to(device)
    return super( VideoLipreading, self ).to( device )

  def forward(self, x):
    x = self.frontend3D(x)
    x = x.transpose(1, 2)
    x = x.contiguous()
    x = x.view(-1, 64, x.size(3), x.size(4))
    x = self.resnet34(x)
    x = x.view(-1, self.frameLen, self.inputDim)
    x = self.gru(x)
    return x

#Evaluación

In [14]:
def CenterCrop(batch_img, size):
  w, h = batch_img[0][0].shape[1], batch_img[0][0].shape[0]
  th, tw = size
  img = np.zeros((len(batch_img), len(batch_img[0]), th, tw))
  for i in range(len(batch_img)):
    x1 = int(round((w - tw))/2.)
    y1 = int(round((h - th))/2.)
    img[i] = batch_img[i, :, y1:y1+th, x1:x1+tw]
  return img

def ColorNormalize(batch_img):
  mean = 0.413621
  std = 0.1700239
  batch_img = (batch_img - mean) / std
  return batch_img

In [15]:
def reload_model(model, path=""):
  model_dict = model.state_dict()
  pretrained_dict = torch.load(path)
  pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict}
  model_dict.update(pretrained_dict)
  model.load_state_dict(model_dict)
  print('*** model has been successfully loaded! ***')
  return model

In [16]:
device = torch.device( 'cuda' if torch.cuda.is_available() else 'cpu' )
print( 'running on: %s' % (device) )

every_frame = True
audio_model = AudioLipreading(inputDim=512, hiddenDim=512, nClasses=500, frameLen=29)
video_model = VideoLipreading(inputDim=256, hiddenDim=512, nClasses=500, frameLen=29)
concat_model = GRU(2048, 512, 2, 500, output_layer=True, every_frame=every_frame)

# reload model
print('reload audio model')
audio_model = reload_model(audio_model, 'Audiovisual/Audiovisual_a_part.pt')
print("reload video model")
video_model = reload_model(video_model, 'Audiovisual/Audiovisual_v_part.pt')
print("reload LSTM model")
concat_model = reload_model(concat_model, 'Audiovisual/Audiovisual_c_part.pt')

audio_model = audio_model.to( device )
video_model = video_model.to( device )
concat_model = concat_model.to( device )

running on: cuda
reload audio model
*** model has been successfully loaded! ***
reload video model
*** model has been successfully loaded! ***
reload LSTM model
*** model has been successfully loaded! ***


In [19]:
import torch

for nombre, modelo, ruta in [
    ('audio',  audio_model,  'Audiovisual/Audiovisual_a_part.pt'),
    ('video',  video_model,  'Audiovisual/Audiovisual_v_part.pt'),
    ('fusión', concat_model, 'Audiovisual/Audiovisual_c_part.pt'),
]:
    ckpt = torch.load(ruta, map_location='cpu')
    md = modelo.state_dict()
    coinciden = [k for k in ckpt if k in md]
    sobran    = [k for k in ckpt if k not in md]   # descartadas en silencio
    faltan    = [k for k in md if k not in ckpt]   # quedaron aleatorias
    n = sum(p.numel() for p in modelo.parameters())
    print(f'{nombre:7s} | {len(coinciden):3d}/{len(ckpt):3d} claves cargadas | '
          f'{len(sobran):2d} descartadas | {len(faltan):2d} sin cargar | {n:>11,} params')
    if faltan:
        print('          ⚠️ sin cargar:', faltan[:5])

    # ¿los pesos son realmente los del checkpoint y no ruido de init?
    k0 = coinciden[0]
    print(f'          muestra "{k0}": ¿idéntico al checkpoint? '
          f'{torch.allclose(md[k0].cpu(), ckpt[k0].cpu())}')


audio   | 120/156 claves cargadas | 36 descartadas | 20 sin cargar |  12,500,340 params
          ⚠️ sin cargar: ['fronted1D.1.num_batches_tracked', 'resnet18.layer1.0.bn1.num_batches_tracked', 'resnet18.layer1.0.bn2.num_batches_tracked', 'resnet18.layer1.1.bn1.num_batches_tracked', 'resnet18.layer1.1.bn2.num_batches_tracked']
          muestra "fronted1D.0.weight": ¿idéntico al checkpoint? True
video   | 204/240 claves cargadas | 36 descartadas | 37 sin cargar |  29,025,460 params
          ⚠️ sin cargar: ['frontend3D.1.num_batches_tracked', 'resnet34.layer1.0.bn1.num_batches_tracked', 'resnet34.layer1.0.bn2.num_batches_tracked', 'resnet34.layer1.1.bn1.num_batches_tracked', 'resnet34.layer1.1.bn2.num_batches_tracked']
          muestra "frontend3D.0.weight": ¿idéntico al checkpoint? True
fusión  |  18/ 18 claves cargadas |  0 descartadas |  0 sin cargar |  13,107,700 params
          muestra "gru.weight_ih_l0": ¿idéntico al checkpoint? True


In [17]:
dset_loader, dset_size = data_loader('test', 'preprocessed/audio', 'preprocessed/video', batch_size = 1)


Statistics: test: 2500


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [20]:
import torch

for nombre, modelo, ruta in [
    ('audio', audio_model, 'Audiovisual/Audiovisual_a_part.pt'),
    ('video', video_model, 'Audiovisual/Audiovisual_v_part.pt'),
]:
    ckpt = torch.load(ruta, map_location='cpu')
    md = modelo.state_dict()
    sobran = [k for k in ckpt if k not in md]

    n_params = sum(ckpt[k].numel() for k in sobran)
    print(f'\n=== {nombre}: {len(sobran)} claves descartadas '
          f'({n_params:,} parámetros muertos en el archivo) ===')
    for k in sobran:
        print(f'  {k:50s} {tuple(ckpt[k].shape)}')



=== audio: 36 claves descartadas (23,344,616 parámetros muertos en el archivo) ===
  backend_conv1.0.weight                             (1024, 512, 5)
  backend_conv1.1.weight                             (1024,)
  backend_conv1.1.bias                               (1024,)
  backend_conv1.1.running_mean                       (1024,)
  backend_conv1.1.running_var                        (1024,)
  backend_conv1.4.weight                             (2048, 1024, 5)
  backend_conv1.5.weight                             (2048,)
  backend_conv1.5.bias                               (2048,)
  backend_conv1.5.running_mean                       (2048,)
  backend_conv1.5.running_var                        (2048,)
  backend_conv2.0.weight                             (512, 2048)
  backend_conv2.0.bias                               (512,)
  backend_conv2.1.weight                             (512,)
  backend_conv2.1.bias                               (512,)
  backend_conv2.1.running_mean                

In [18]:
audio_model.eval()
video_model.eval()
concat_model.eval()

running_loss = 0.0
running_corrects = 0.0
running_all = 0.0
with torch.no_grad():
  for batch_idx, (audio_inputs, video_inputs, targets) in enumerate(dset_loader):
    batch_img = CenterCrop(video_inputs.numpy(), (88, 88))
    batch_img = ColorNormalize(batch_img)

    batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
    video_inputs = torch.from_numpy(batch_img)
    video_inputs = video_inputs.float().permute(0, 4, 1, 2, 3)

    audio_inputs = audio_inputs.float()

    audio_inputs = audio_inputs.to( device )
    video_inputs = video_inputs.to( device )
    targets = targets.to( device )

    audio_outputs = audio_model(audio_inputs)
    video_outputs = video_model(video_inputs)
    inputs = torch.cat((audio_outputs, video_outputs), dim=2)
    outputs = concat_model(inputs)

    if every_frame:
      outputs = torch.mean(outputs, 1) # average probability among frames
    _, preds = torch.max(F.softmax(outputs, dim=1).data, 1)

    #running_loss += loss.data[0] * inputs.size(0)
    running_corrects += torch.sum(preds == targets.data)
    running_all += len(inputs)
print('Accuracy: {:.4f}'.format(running_corrects / len(dset_loader.dataset))+'\n')

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Accuracy: 0.9884



In [21]:
import torch, torch.nn.functional as F, numpy as np

with open('label_sorted.txt') as f:
    id2label = f.read().splitlines()

audio_model.eval(); video_model.eval(); concat_model.eval()
errores = []
loader, _ = data_loader('test', 'preprocessed/audio', 'preprocessed/video', batch_size=1)

with torch.no_grad():
    for a_in, v_in, tgt in loader:
        img = ColorNormalize(CenterCrop(v_in.numpy(), (88, 88)))
        img = np.reshape(img, (*img.shape, 1))
        v = torch.from_numpy(img).float().permute(0, 4, 1, 2, 3).to(device)
        a = a_in.float().to(device)
        out = concat_model(torch.cat((audio_model(a), video_model(v)), dim=2))
        out = torch.mean(out, 1)
        p = F.softmax(out, dim=1)
        conf, pred = torch.max(p, 1)
        if pred.item() != tgt.item():
            errores.append((id2label[tgt.item()], id2label[pred.item()], conf.item()))

print(f'{len(errores)} errores de 2500 → accuracy {1 - len(errores)/2500:.4f}\n')
print(f'{"real":<16}{"predicho":<16}{"confianza":>10}')
for real, pred, c in sorted(errores, key=lambda e: -e[2]):
    print(f'{real:<16}{pred:<16}{c:>10.3f}')



Statistics: test: 2500
29 errores de 2500 → accuracy 0.9884

real            predicho         confianza
MEETING         MAKING               0.996
ELECTION        ACTION               0.980
WORLD           WHILE                0.979
SPEND           SPENT                0.969
ELECTION        ACTION               0.961
WORDS           WOULD                0.929
BORDER          IMPORTANT            0.928
POSITION        OPPOSITION           0.927
EXPECT          EXPECTED             0.891
TAKEN           TAKING               0.890
ALLOW           WITHOUT              0.879
QUESTIONS       QUESTION             0.848
COMPANY         COMPANIES            0.782
BENEFITS        BENEFIT              0.781
PLACES          PRICES               0.759
ASKED           ANSWER               0.742
WORST           WORDS                0.741
PHONE           THIRD                0.738
HAPPENED        HAPPEN               0.642
WORDS           WORST                0.616
REASON          RECENT             

##Evaluación cualitativa

In [27]:
def normalisation(inputs):
  inputs_std = np.std(inputs)
  if inputs_std == 0.:
    inputs_std = 1.
  return (inputs - np.mean(inputs))/inputs_std

def predict(filename):
  # data loading
  npy_basename = 'preprocessed'
  with open('label_sorted.txt') as myfile:
    id2label = myfile.read().splitlines()
    label2id = { klass:i for i, klass in enumerate(id2label) }
  audio_input = load_audio_file( os.path.join( npy_basename,
                                               'audio',
                                                filename.split('/')[-3],
                                                filename.split('/')[-2],
                                                filename.split('/')[-1][:-4]+'.npz' ) )
  audio_input = normalisation(audio_input)
  video_input = load_video_file( os.path.join( npy_basename,
                                               'video',
                                                filename.split('/')[-3],
                                                filename.split('/')[-2],
                                                filename.split('/')[-1][:-4]+'.npz' ) )
  label = filename.split('/')[-3]
  label_id = label2id[label]

  audio_input = np.expand_dims( audio_input, 0 ) # add batch dimension
  video_input = np.expand_dims( video_input, 0 ) # add batch dimension

  # prediction
  batch_img = CenterCrop(video_input, (88, 88))
  batch_img = ColorNormalize(batch_img)
  batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
  video_input = torch.from_numpy(batch_img)
  video_input = video_input.float().permute(0, 4, 1, 2, 3)


  audio_input = torch.from_numpy(audio_input).float()

  audio_input = audio_input.to( device )
  video_input = video_input.to( device )

  audio_output = audio_model(audio_input)
  video_output = video_model(video_input)
  input = torch.cat((audio_output, video_output), dim=2)
  output = concat_model(input)

  if every_frame:
    output = torch.mean(output, 1) # average probability among frames
  _, pred = torch.max(F.softmax(output, dim=1).data, 1)

  pred_str = id2label[int(pred)]

  return pred, pred_str

**Corrección aplicada a la función `predict`**

La versión original terminaba con `return preds, pred_str`, pero la variable local
calculada dentro de la función es `pred` (sin `s`). Como `preds` no existe en el ámbito
local, Python la resolvía en el ámbito global, donde encontraba la variable remanente
del loop de evaluación de la sección anterior:

```python
# en el loop de evaluación
_, preds = torch.max(F.softmax(outputs, dim=1).data, 1)   # sobrevive al for
```

En consecuencia, `predict` devolvía la predicción del último clip procesado por el loop
de evaluación, no la del video solicitado. El efecto era sutil porque `pred_str` sí se
calculaba correctamente: la palabra impresa era la correcta, pero el índice numérico que
la acompañaba pertenecía a otra muestra. Además, como el `DataLoader` usa `shuffle=True`,
el índice cambiaba en cada ejecución, y sin haber corrido antes el loop de evaluación la
función habría fallado con `NameError`.

Se corrigió a `return pred, pred_str`. Verificación: para
`EXAMPLE/test/EXAMPLE_00001.mp4` la función devuelve ahora el índice **146**, que
coincide con la posición de `EXAMPLE` en `label_sorted.txt`.


In [30]:
#video_filename = 'lipread_testset_mini/AMERICAN/test/AMERICAN_00001.mp4'
#ideo_filename = 'lipread_testset_mini/CHILDREN/test/CHILDREN_00001.mp4'
video_filename = 'lipread_testset_mini/EXAMPLE/test/EXAMPLE_00001.mp4'
#video_filename = 'lipread_testset_mini/MAKING/test/MAKING_00001.mp4'
#video_filename = 'lipread_testset_mini/YESTERDAY/test/YESTERDAY_00001.mp4'

pred, pred_str = predict(video_filename)
print( 'Model prediction: %s [%d]' % (pred_str, int(pred)) )

Model prediction: EXAMPLE [146]


In [31]:
from IPython.display import HTML
from base64 import b64encode
# Convert mp4 to format supported by Colab
os.system(f"ffmpeg -i {video_filename} -vcodec libx264 {os.path.basename(video_filename)}")
mp4 = open(os.path.basename(video_filename),'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<h1>Predicted word: %s</h1>
<br>
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % (pred_str, data_url))

#Actividades

##Actividad 1

¿ Por qué puede ser necesario utilizar los frames de video para la tarea de speech recognition ?

In [ ]:
Respuesta = 'Para dar robustez al modelo cuando el audio viene con ruido ambiente' #@param ["seleccione una opcion", "Los frames de video son utilizados para leer los labios y generar el audio del habla","Para dar robustez al modelo cuando el audio viene con ruido ambiente", "Para localizar la persona que está hablando dentro de la imagen", "Para determinar el intervalo de tiempo donde se produce el habla", "Los frames de video son imprescindibles para reconocer la palabra pronunciada"]

**Justificación**

La Tabla 1 del paper muestra que con audio limpio el aporte del video es marginal:
el modelo de solo audio alcanza 97,7 % y el audiovisual 98,0 % (+0,3 puntos). Sin
embargo, la Figura 3 mide el desempeño en función del ruido y ahí el panorama cambia
por completo:

| Condición | Ganancia del modelo audiovisual sobre el de solo audio |
|-----------|--------------------------------------------------------|
| 5 dB SNR  | +1,3 puntos  |
| 0 dB SNR  | +3,9 puntos  |
| −5 dB SNR | **+14,1 puntos** |

El paper es explícito: *"the contribution of the visual modality is usually marginal
in clean audio conditions"*, pero *"it significantly outperforms both of them under
high noise levels"*. El video no está para mejorar el caso fácil, sino para sostener
el caso degradado.

Esto se refleja en el propio código: la constante `self.clean = 1/7.` de `MyDataset`
corresponde a las siete condiciones equiprobables con que se entrena según la sección
4.3 del paper — audio limpio más babble noise a 20, 15, 10, 5, 0 y −5 dB.


##Actividad 2

¿ Por qué el entrenamiento del modelo se hace por etapas ? ( primero ResNets, luego BiGRUs, luego todo junto ).

In [ ]:
Respuesta = 'Para aumentar la estabilidad del entrenamiento y obtener un mayor rendimiento' #@param ["seleccione una opcion", "Porque el modelo es muy grande y no es posible alamcenar el gradiente de todos sus pesos en una GPU","Porque no es posible combinar modelos feedforward con modelos recurrentes en la propagación de gradientes", "Para separar el entrenamiento del stream de video del de audio", "Para aumentar la estabilidad del entrenamiento y obtener un mayor rendimiento", "Para separar cada uno de los 29 instantes de tiempo que componen los videos de entrada"]

**Justificación**

El paper lo afirma textualmente en la sección 4.3.1: *"**Directly training end-to-end
each stream leads to suboptimal performance** so we follow the same 3-step procedure."*

El procedimiento completo es:

1. **Fase 1** — se entrena cada ResNet con un *backend temporal convolucional* y una
   capa softmax, hasta que la tasa de clasificación en validación deja de mejorar.
2. **Fase 2** — se remueve ese backend, se conecta el BiGRU de 2 capas y se entrena
   solo el BiGRU durante 5 épocas, con la ResNet congelada.
3. **Fase 3** — se destraba todo y se entrena end-to-end con Adam (lr 3e-4, batch 36
   por stream; lr 1e-4, batch 18 para el modelo audiovisual completo).

La razón de fondo es de optimización: una red recurrente montada sobre una ResNet sin
entrenar converge mal, porque el gradiente que llega a las capas convolucionales debe
atravesar 29 pasos temporales de compuertas. Un backend convolucional es mucho más
fácil de optimizar y sirve para estabilizar primero el extractor de features.

**Evidencia empírica en los checkpoints**

Al inspeccionar los `state_dict` descargados se encuentran 36 claves que el modelo
actual nunca utiliza, y que corresponden justamente a las etapas descartadas:

```
backend_conv1.0.weight   (1024, 512, 5)   ← backend temporal-convolucional (fase 1)
backend_conv2.3.weight   (500, 512)       ← su capa softmax de 500 clases
lstm.forwardModule1...   (2048, 512)      ← BiLSTM de 2 capas de una versión anterior
```

Son 23,3 M de parámetros inertes en el checkpoint de audio (el 65 % del archivo) y
11,5 M en el de video. El backend fue "removido" del grafo de cómputo, pero quedó
grabado en el archivo: es el registro físico del entrenamiento por etapas.


##Actividad 3

Como vimos en clases, para la *rama* que se encarga de procesar el video, se comienza por agregar una capa convolucional 3D. ¿ Cuál es el principal objetivo de esta capa ?

**Hint:** Si no se acuerda, puede descargar el paper y leer la sección 3.1, página 2.

In [ ]:
Respuesta = 'Capturar las dinámicas producidas en pequeños intervalos de tiempo' #@param ["seleccione una opcion", "Reducir la dimensión temporal desde 29 frames a 1 que resuma todo el movimiento","Reducir los 3 canales RGB de entrada a una matriz bidimensional", "Realizar un downsampling de algunos frames que permitan recuperar patrones temporales", "Capturar las dinámicas producidas en pequeños intervalos de tiempo"]

**Justificación**

La configuración de la capa lo demuestra sin necesidad de interpretación:

```python
nn.Conv3d(1, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
```

| Eje | Kernel | Stride | Padding | Transformación |
|-----|--------|--------|---------|----------------|
| Tiempo  | 5 | **1** | **2** | **29 → 29 (sin cambio)** |
| Alto    | 7 | 2 | 3 | 88 → 44 |
| Ancho   | 7 | 2 | 3 | 88 → 44 |

Con stride temporal 1 y padding 2 entran 29 frames y salen 29. El `MaxPool3d` que le
sigue tiene kernel y stride temporal 1, de modo que tampoco altera el eje del tiempo.
La reducción es exclusivamente espacial.

Lo que la capa sí hace es dotar a cada unidad de salida de un campo receptivo de
**5 frames consecutivos**, equivalentes a **200 ms** a 25 fps — aproximadamente la
duración de una sílaba en habla continua. No aprende cómo *se ve* una boca, sino cómo
**se mueve** una boca en un intervalo corto.

El paper lo justifica en la sección 3.1: *"A spatiotemporal convolutional layer is
capable of **capturing the short-term dynamics of the mouth region** and is proven to
be advantageous, **even when recurrent networks are deployed for back-end**."*

Ese "even when" es central: no basta con poner un BiGRU al final. La recurrente modela
la dinámica a escala de la palabra completa (1,16 s), pero opera sobre vectores de 256
números por frame, cuando la información del movimiento rápido ya se perdió. Hay que
capturarla antes, mientras todavía hay píxeles.

Conviene notar también que la ResNet-34 posterior procesa los 29 frames de forma
**independiente** (el `view(-1, 64, 22, 22)` colapsa el eje temporal dentro del batch).
Toda la modelación temporal del stream recae, por tanto, en dos lugares: esta Conv3D
para el corto plazo y el BiGRU para el largo plazo.
